# Cost Management & Security Governance — Demo

Practical cost monitoring with System Tables + Unity Catalog governance: column masking, row-level security, audit logging, and Delta Sharing.

| Training Block | Duration | Type |
|---|---|---|
| Cost Management & Security — Demo | 45 min | Demo |

## Learning Objectives

After completing this module you will be able to:

**Cost Management:**
- **Identify** the main cost drivers (DBU, compute type, storage operations)
- **Monitor** DBU usage and costs using System Tables (`system.billing.*`, `system.compute.*`)
- **Evaluate** the cost impact of OPTIMIZE, caching, and cluster policies
- **Apply** cost management best practices for clusters and pipelines

**Security & Governance:**
- **Explain** Unity Catalog architecture: metastore → catalog → schema → table
- **Implement** column masking and row-level security using `ALTER TABLE … SET MASK`
- **Query** audit logs and data lineage from System Tables
- **Configure** Delta Sharing for secure cross-org data exchange

## Setup

In [ ]:
%run ../../setup/00_setup

## Databricks Cost Model Overview

### Two Main Cost Drivers

| Component | What You Pay For | How to Optimize |
|-----------|------------------|-----------------|
| **Compute (DBUs)** | Time × cluster size × DBU rate | Right-size clusters, use serverless, autoscaling |
| **Storage** | GB stored in cloud storage (S3/ADLS/GCS) | VACUUM, avoid over-partitioning, compression |

> **Rule of thumb:** Compute costs are typically **5-10x higher** than storage costs. Optimize compute first.

### Compute Types & Cost

| Compute Type | Use Case | Cost Efficiency |
|---|---|---|
| **All-Purpose Cluster** | Development, exploration | Most expensive per DBU |
| **Job Cluster** | Production pipelines | ~2x cheaper than all-purpose |
| **SQL Warehouse** | BI queries, dashboards | Serverless, auto-suspends |
| **Serverless** | Variable workloads | Pay only for active time |

### Key insight: Use Job Clusters for production pipelines, not All-Purpose.

## Impact of Optimization on Cost

### OPTIMIZE & Compaction

| Before OPTIMIZE | After OPTIMIZE |
|---|---|
| 5,000 small files (1MB each) | 30 files (128MB each) |
| Slow queries (many file opens) | Fast queries (fewer file reads) |
| Higher compute cost per query | Lower compute cost per query |

**Cost impact:** OPTIMIZE itself costs compute time, but saves much more on downstream queries.

### Partitioning Cost Trade-offs

| Scenario | Impact |
|---|---|
| **Over-partitioning** (e.g., by `customer_id`) | Small files problem → expensive queries |
| **Under-partitioning** (no partitions) | Full table scans → expensive queries |
| **Right partitioning** (e.g., by `year/month`) | Partition pruning → cheaper queries |

> **Best practice:** Use Liquid Clustering instead of manual partitioning for new tables.

### VACUUM & Storage Costs

- Default retention: 7 days (keeps old file versions for time travel)
- Shorter retention = less storage cost BUT less recovery ability
- production recommendation: 7 days for operational tables, 30 days for audit tables

## Cost Architecture in Bronze-Silver-Gold

| Layer | Storage Pattern | Compute Pattern | Cost Tip |
|---|---|---|---|
| **Bronze** | Append-only, largest volume | Streaming/batch ingestion (low compute) | Use Auto Loader (incremental), avoid full reloads |
| **Silver** | MERGE/CDC, moderate volume | Transformation (medium compute) | Use `availableNow` trigger for batch, incremental only |
| **Gold** | Aggregated, smallest volume | Heavy joins & aggregations (high compute per row) | Use MATERIALIZED VIEWs, cache frequent queries |

### Cost Optimization Strategies by Layer

**Bronze:**
- Auto Loader with `availableNow` trigger (process only new files, then stop)
- Avoid schema inference at scale (define explicit schemas)
- Use Liquid Clustering over partitioning

**Silver:**
- Incremental processing only (MERGE, not full overwrite)
- Right-size compute: don't use a 16-node cluster for a 100K row MERGE
- Schedule during off-peak hours (cheaper spot instances)

**Gold:**
- MATERIALIZED VIEWs auto-refresh only when source changes
- Photon-enabled clusters for SQL-heavy aggregations
- Pre-aggregate common queries (avoid repeated expensive joins)

### Serverless Compute Cost Model

Serverless compute (GA 2025) uses a simplified pricing model:

| Aspect | Classic Compute | Serverless Compute |
|--------|----------------|-------------------|
| **Billing** | DBU + cloud VM cost | DBU only (all-inclusive) |
| **Idle cost** | Cluster stays running | Auto-scales to zero |
| **Spot savings** | Available | Not applicable |
| **Best for** | Long-running, predictable | Short-burst, event-driven |

> **Pro Tip:** Monitor serverless DBU consumption in `system.billing.usage` where `sku_name` contains 'SERVERLESS'. Compare cost-per-job between serverless and classic to find the optimal mix.

## Practical Cost Monitoring

### System Tables for Cost Tracking

```sql
-- DBU usage by cluster (last 7 days)
SELECT
 cluster_id,
 SUM(usage_quantity) AS total_dbus,
 usage_type
FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -7, CURRENT_DATE())
GROUP BY cluster_id, usage_type
ORDER BY total_dbus DESC;
```

```sql
-- Storage size by table
SELECT
 table_catalog, table_schema, table_name,
 ROUND(data_size_bytes / 1024 / 1024 / 1024, 2) AS size_gb
FROM system.information_schema.tables
WHERE table_catalog = '${CATALOG}'
ORDER BY size_gb DESC;
```

## Monitoring & Observability (System Tables)

Unity Catalog provides **System Tables** (`system.*`) for operational monitoring and observability.
These tables give insights into costs, job runs, pipeline health, query performance, and storage usage.

**Available System Table Categories:**

| Category | Schema | Key Tables |
|----------|--------|------------|
| **Billing** | `system.billing` | `usage`, `list_prices` |
| **Compute** | `system.compute` | `clusters`, `warehouse_events` |
| **Workflows** | `system.lakeflow` | `job_run_timeline`, `job_task_run_timeline` |
| **Pipelines** | `system.lakeflow` | `pipeline_event_log`, `pipeline_update_timeline` |
| **Queries** | `system.query` | `history` |
| **Storage** | `system.storage` | `predictive_optimization_operations_history` |
| **Access** | `system.access` | `audit`, `table_lineage`, `column_lineage` |

> **Note**: System tables require **Metastore admin** or specific `MONITOR` privileges.

**System Tables — Quick Reference**

| Table | What It Tracks |
|-------|---------------|
| `system.billing.usage` | DBU consumption by workspace, SKU, cluster, date |
| `system.query.history` | SQL query history — duration, user, warehouse, status |
| `system.compute.clusters` | Cluster events, uptime, compute type |
| `system.access.audit` | Unity Catalog access, permission changes, login events |
| `system.access.table_lineage` | Column-level data lineage across tables/views |
| `system.lakeflow.job_run_timeline` | Job run history, duration, success/failure |
| `system.lakeflow.pipeline_event_log` | Lakeflow pipeline events and data quality metrics |

### Cost Monitoring (DBU Usage)

Track Databricks Unit (DBU) consumption by workspace, SKU, and user. Essential for budget management and chargeback.

In [0]:
# Daily DBU cost breakdown by SKU (last 30 days)
# NOTE: Requires Metastore Admin or MONITOR privilege on system.billing
try:
    cost_daily = spark.sql("""
        SELECT 
            usage.usage_date,
            usage.sku_name,
            usage.usage_unit,
            SUM(usage.usage_quantity) as total_dbus,
            ROUND(SUM(usage.usage_quantity * list_prices.pricing.default), 2) as estimated_cost_usd
        FROM system.billing.usage
        LEFT JOIN system.billing.list_prices 
            ON usage.sku_name = list_prices.sku_name
            AND usage.usage_date BETWEEN list_prices.price_start_time AND COALESCE(list_prices.price_end_time, '2099-12-31')
        WHERE usage.usage_date >= current_date() - INTERVAL 30 DAYS
        GROUP BY usage.usage_date, usage.sku_name, usage.usage_unit
        ORDER BY usage.usage_date DESC, estimated_cost_usd DESC
    """)
    display(cost_daily)
except Exception as e:
    print(f"⚠ Cannot access system.billing.usage — requires Metastore Admin or MONITOR privilege.")
    print(f"  Error: {e}")
    print("  Instructor: run this query in a workspace where you have admin access.")


In [0]:
# Top 10 most expensive users (last 30 days)
try:
    cost_by_user = spark.sql("""
        SELECT 
            identity_metadata.run_as as run_as_user,
            sku_name,
            ROUND(SUM(usage_quantity), 2) as total_dbus,
            COUNT(DISTINCT usage_date) as active_days
        FROM system.billing.usage
        WHERE usage_date >= current_date() - INTERVAL 30 DAYS
            AND identity_metadata.run_as IS NOT NULL
        GROUP BY identity_metadata.run_as, sku_name
        ORDER BY total_dbus DESC
        LIMIT 10
    """)
    display(cost_by_user)
except Exception as e:
    print(f"⚠ Cannot access system.billing.usage — requires Metastore Admin.")
    print(f"  Error: {e}")


In [0]:
# Cost trend: weekly aggregation with week-over-week change
try:
    cost_trend = spark.sql("""
        WITH weekly AS (
            SELECT
                DATE_TRUNC('week', usage_date) as week_start,
                ROUND(SUM(usage_quantity), 2) as total_dbus
            FROM system.billing.usage
            WHERE usage_date >= current_date() - INTERVAL 90 DAYS
            GROUP BY DATE_TRUNC('week', usage_date)
        )
        SELECT
            week_start,
            total_dbus,
            LAG(total_dbus) OVER (ORDER BY week_start) as prev_week_dbus,
            ROUND(
                (total_dbus - LAG(total_dbus) OVER (ORDER BY week_start)) 
                / LAG(total_dbus) OVER (ORDER BY week_start) * 100, 1
            ) as wow_change_pct
        FROM weekly
        ORDER BY week_start DESC
    """)
    display(cost_trend)
except Exception as e:
    print(f"⚠ Cannot access system.billing.usage — requires Metastore Admin.")
    print(f"  Error: {e}")


### Job & Workflow Monitoring

Monitor Lakeflow Jobs execution: success rates, durations, failures. Critical for SLA compliance.

In [0]:
# Failed jobs with error details (last 24 hours)
failed_jobs = spark.sql("""
    SELECT
        job_id,
        run_name,
        run_id,
        result_state,
        period_start_time as start_time,
        period_end_time as end_time,
        TIMESTAMPDIFF(MINUTE, period_start_time, period_end_time) as duration_min,
        run_type
    FROM system.lakeflow.job_run_timeline
    WHERE result_state IN ('FAILED', 'TIMEDOUT', 'CANCELED')
        AND period_start_time >= current_timestamp() - INTERVAL 24 HOURS
    ORDER BY period_start_time DESC
""")

display(failed_jobs)

### Lakeflow Pipeline Monitoring

Track Lakeflow Declarative Pipeline health, update durations, and data quality issues.

In [0]:
# Note: system.lakeflow.pipeline_event_log is not available in this environment
# Using job_run_timeline instead to demonstrate system table monitoring
job_runs = spark.sql("""
    SELECT
        job_id,
        run_id,
        run_name,
        result_state,
        period_start_time,
        period_end_time,
        run_type
    FROM system.lakeflow.job_run_timeline
    WHERE period_start_time >= current_date() - INTERVAL 7 DAYS
    ORDER BY period_start_time DESC
    LIMIT 50
""")

display(job_runs)

In [0]:
# Note: system.lakeflow.pipeline_update_timeline is not available in this environment
# Using job_task_run_timeline instead to demonstrate task-level monitoring
task_health = spark.sql("""
    SELECT
        job_id,
        task_key,
        run_id,
        result_state,
        COUNT(*) as total_runs,
        SUM(CASE WHEN result_state = 'SUCCESS' THEN 1 ELSE 0 END) as successes,
        SUM(CASE WHEN result_state IN ('FAILED', 'TIMEDOUT', 'CANCELED') THEN 1 ELSE 0 END) as failures,
        ROUND(AVG(TIMESTAMPDIFF(MINUTE, period_start_time, period_end_time)), 1) as avg_duration_min,
        MAX(period_start_time) as last_run_time
    FROM system.lakeflow.job_task_run_timeline
    WHERE period_start_time >= current_date() - INTERVAL 30 DAYS
    GROUP BY job_id, task_key, run_id, result_state
    ORDER BY failures DESC, total_runs DESC
    LIMIT 50
""")

display(task_health)

### Query Performance Monitoring

Identify slow queries, heavy users, and optimization opportunities using SQL Warehouse query history.

In [0]:
# Slowest queries (last 7 days, > 60s)
slow_queries = spark.sql("""
    SELECT
        statement_id,
        executed_by as user,
        SUBSTRING(statement_text, 1, 120) as query_preview,
        execution_status,
        total_duration_ms / 1000 as duration_sec,
        produced_rows,
        start_time
    FROM system.query.history
    WHERE start_time >= current_date() - INTERVAL 7 DAYS
        AND total_duration_ms > 60000
        AND statement_type IN ('SELECT', 'MERGE', 'INSERT', 'CREATE_TABLE_AS_SELECT')
    ORDER BY total_duration_ms DESC
    LIMIT 20
""")

display(slow_queries)

In [0]:
# Query volume and performance by user (last 7 days)
query_by_user = spark.sql("""
    SELECT
        executed_by as user,
        COUNT(*) as total_queries,
        ROUND(AVG(total_duration_ms / 1000), 2) as avg_duration_sec,
        ROUND(MAX(total_duration_ms / 1000), 2) as max_duration_sec,
        SUM(produced_rows) as total_rows_produced,
        COUNT(CASE WHEN execution_status = 'FAILED' THEN 1 END) as failed_queries
    FROM system.query.history
    WHERE start_time >= current_date() - INTERVAL 7 DAYS
    GROUP BY executed_by
    ORDER BY total_queries DESC
    LIMIT 15
""")

display(query_by_user)

### Compute & Cluster Monitoring

Track cluster utilization, uptime, and idle time to optimize compute costs.

In [0]:
# Active clusters with uptime and DBU usage
cluster_usage = spark.sql("""
    SELECT
        cluster_id,
        cluster_name,
        cluster_source,
        driver_node_type,
        worker_count,
        change_time,
        dbr_version
    FROM system.compute.clusters
    WHERE change_time >= current_date() - INTERVAL 7 DAYS
    ORDER BY change_time DESC
    LIMIT 50
""")

display(cluster_usage)

In [0]:
# SQL Warehouse usage patterns
warehouse_usage = spark.sql("""
    SELECT
        warehouse_id,
        event_type,
        event_time,
        cluster_count
    FROM system.compute.warehouse_events
    WHERE event_time >= current_date() - INTERVAL 7 DAYS
    ORDER BY event_time DESC
    LIMIT 50
""")

display(warehouse_usage)

### Storage & Table Size Monitoring

Monitor table sizes, growth trends, and identify tables that need optimization.

In [0]:
# Note: system.storage.table_storage is not available in this environment
# Using information_schema.tables to list tables in the catalog
# For detailed storage metrics, use DESCRIBE DETAIL or ANALYZE TABLE COMPUTE STORAGE METRICS on individual tables

tables_list = spark.sql(f"""
    SELECT
        table_catalog,
        table_schema,
        table_name,
        table_type,
        table_owner,
        last_altered
    FROM {CATALOG}.information_schema.tables
    WHERE table_type IN ('MANAGED', 'EXTERNAL')
    ORDER BY last_altered DESC
    LIMIT 20
""")

display(tables_list)

In [0]:
# Predictive Optimization history
pred_opt = spark.sql(f"""
    SELECT
        catalog_name,
        schema_name,
        table_name,
        operation_type,
        operation_status,
        usage_quantity as dbus_used,
        start_time,
        end_time
    FROM system.storage.predictive_optimization_operations_history
    WHERE catalog_name = '{CATALOG}'
        AND start_time >= current_date() - INTERVAL 30 DAYS
    ORDER BY start_time DESC
    LIMIT 30
""")

display(pred_opt)

### Governance Health Dashboard

Combined governance health check: tables without comments, untagged PII, permissions audit.

In [0]:
# Tables without comments (governance gap)
undocumented = spark.sql(f"""
    SELECT 
        table_catalog,
        table_schema,
        table_name,
        table_type
    FROM {CATALOG}.information_schema.tables
    WHERE comment IS NULL OR comment = ''
    ORDER BY table_schema, table_name
""")

display(undocumented)

In [0]:
# All tags across catalog (governance inventory)
all_tags = spark.sql(f"""
    SELECT 
        catalog_name,
        schema_name,
        table_name,
        column_name,
        tag_name,
        tag_value
    FROM system.information_schema.column_tags
    WHERE catalog_name = '{CATALOG}'
    ORDER BY schema_name, table_name, column_name
""")

display(all_tags)

In [0]:
# Permission audit: all grants in catalog
perm_audit = spark.sql(f"""
    SELECT 
        grantor,
        grantee,
        table_catalog,
        table_schema,
        table_name,
        privilege_type,
        is_grantable
    FROM {CATALOG}.information_schema.table_privileges
    ORDER BY grantee, table_schema, table_name
""")

display(perm_audit)

> **Tip**: Create a **Lakeflow Job** that runs these monitoring queries daily and sends alerts via email/Slack on anomalies (e.g., cost spike > 20%, job failure rate > 5%, tables without comments).

## Cost Management Best Practices (Databricks Admin Essentials)

Source reference: Databricks blog, *Best Practices for Cost Management on Databricks*.

### 1) Put Guardrails First (Cluster Policies)

- Restrict unrestricted cluster creation entitlement where possible.
- Enforce sensible limits: max workers, autoscaling, auto-termination.
- Use `cluster_type` policies to separate use cases:
  - **job** for automated pipelines
  - **all-purpose** for interactive exploration
- Use `dbus_per_hour` to cap runaway compute profiles.

### 2) Use the Right Compute for the Right Workload

- Prefer **Job Compute** over all-purpose for scheduled ETL (better isolation and cost profile).
- Use **SQL Warehouse** for BI concurrency and external tools (Power BI/Tableau).
- Enable **Photon** and modern runtimes for better price/performance.
- For variable BI traffic, prefer **Serverless SQL Warehouse** when available.

### 3) Improve Cost Attribution and Accountability

- Enforce cost-center/team tags at policy level.
- Standardize mandatory tags (`cost_center`, `team`, `environment`, `owner`).
- Review spend by SKU + tag in account-level dashboards.

### 4) Control Non-DBU Cloud Costs Too

- Storage: align lifecycle policies with Delta `VACUUM` retention.
- Networking: minimize cross-region traffic and use private endpoints where possible.
- Avoid archive tiers for active Delta workloads before safe retention windows.

### 5) Operationalize Monitoring and Alerts

- Daily ingestion of usage/billing exports into Delta + SQL dashboarding.
- Alerts for anomalies (cost spike %, idle clusters, failed jobs, long queries).
- Budget thresholds per workspace/team/SKU for proactive notifications.

> **Trainer framing:** Cost optimization is not only "smaller clusters". It is a balance of guardrails, workload-fit compute, and continuous monitoring.

## Cost Decision Matrix

| Decision | Cheap Option | Expensive Option | When to Spend More |
|---|---|---|---|
| Cluster type | Job cluster (auto-terminate) | All-purpose (always on) | Interactive development only |
| Processing | Incremental (MERGE, Auto Loader) | Full reload (overwrite) | Schema migration, data corruption fix |
| Storage | 7-day VACUUM | 90-day retention | Compliance/audit requirements |
| Clustering | Liquid Clustering | Manual Z-ORDER jobs | Never (Liquid Clustering is preferred) |
| Caching | No caching | `CACHE SELECT` | Repeated identical queries in Gold |
| Format | Delta (compressed Parquet) | JSON/CSV | Never in production |

## Bonus Demo (Optional): SQL Warehouse + Power BI

### Goal

Show a practical daily operating model in Lakehouse projects:

- **Job Compute** for Bronze/Silver/Gold transformations
- **SQL Warehouse** for BI serving and concurrent SQL analytics
- **Power BI** as a business-facing consumption layer on top of SQL Warehouse

### 10-Minute Demo Flow

1. Run or reference a completed transformation task on **Job Compute** (e.g., `fact_sales` refresh).
2. Create a BI-ready Gold view:

```sql
CREATE OR REPLACE VIEW <catalog>.<schema>.v_sales_daily AS
SELECT
  order_date,
  store_id,
  SUM(total_amount) AS revenue,
  COUNT(*) AS orders_count
FROM <catalog>.<schema>.fact_sales
GROUP BY order_date, store_id;
```

3. Execute the same analytical query from **SQL Warehouse** and discuss startup/concurrency behavior.
4. Open **Power BI Desktop** -> Get Data -> Azure -> Azure Databricks, then configure:
   - Server Hostname: workspace hostname
   - HTTP Path: SQL Warehouse endpoint
   - Authentication: Microsoft Entra ID
5. Compare semantic modes:
   - **DirectQuery**: fresher data, higher query latency/cost sensitivity
   - **Import**: faster visuals, scheduled refresh, lower interactive load

### Decision Shortcut

| Daily Need | Recommended Compute |
|---|---|
| Scheduled ingestion, MERGE, Lakeflow pipelines | Job Compute |
| BI dashboards, many concurrent SQL users | SQL Warehouse |
| External BI tools (Power BI/Tableau) | SQL Warehouse |
| Mixed pipeline + BI | Job Compute for ETL + SQL Warehouse for serving |

> **Trainer note:** Emphasize separation of concerns: transform on Job Compute, serve on SQL Warehouse.

## Part 2 — Security & Unity Catalog Governance

Unity Catalog governance: column masking, row-level security, audit logging, and Delta Sharing.

### Configuration
Define paths to data source files used throughout this notebook.

In [0]:
# Paths to data directories (subdirectories in DATASET_PATH from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
ORDERS_PATH = f"{DATASET_PATH}/orders"
PRODUCTS_PATH = f"{DATASET_PATH}/products"

# Paths to specific files
CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"
ORDERS_JSON = f"{ORDERS_PATH}/orders_batch.json"
PRODUCTS_PARQUET = f"{PRODUCTS_PATH}/products.parquet"

## Unity Catalog Architecture

**Unity Catalog** is a unified governance solution for Databricks Lakehouse.

### Object Hierarchy:

```
Metastore (region-level)
 ↓
Catalog (database/domain)
 ↓
Schema (namespace)
 ↓
Securable Objects:
 - Tables / Views
 - Functions (UDF, stored procedures)
 - Volumes (files storage)
 - Models (ML models)
```

### Three-level namespace:
```sql
catalog.schema.table
```

Example:
```sql
main.sales.orders
dev.analytics.customer_metrics
prod.gold.daily_revenue
```

### Key Features:
- **Unified governance**: single platform for data, ML, BI
- **Fine-grained access control**: table, column, row level
- **Automatic lineage**: end-to-end data flow tracking
- **Audit logging**: who accessed what and when
- **Data discovery**: metadata search and tagging

### Setup and Basic Operations
Initialize Unity Catalog objects and verify the working environment.

### Creating User Groups
We create user groups for permission demonstration:
- `data_engineers`: Full access to Bronze/Silver schemas
- `data_analysts`: Read-only access to Gold

In [0]:
# Verification of created schemas
schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").select("databaseName").collect()
schema_names = [row.databaseName for row in schemas]

**Active catalog and schema set**

We set the default working context - all subsequent operations will be executed in this catalog and schema unless a full path is specified.

In [0]:
# Verification of created schemas
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").display()

## Data Preparation

Before we proceed to access management, we will load real data from the dataset/ directory that we will use in the Unity Catalog examples.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.orders")

orders_df = spark.read.option("header", "true").option("inferSchema", "true").json(ORDERS_JSON)
orders_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders")

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")

customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv(CUSTOMERS_CSV)
customers_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.products")

products_df = spark.read.parquet(PRODUCTS_PARQUET)
products_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.products")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products"))

In [0]:
# Verification of orders record count
spark.sql(f"SELECT COUNT(*) as count FROM {CATALOG}.{BRONZE_SCHEMA}.orders").display()

### Demo: Comments and Tags

You can add descriptive comments to Unity Catalog tables and columns using SQL commands. This improves data discoverability and governance.

The cell below demonstrates how to add comments to a table and a specific column using Spark SQL.

In [0]:
# Add comments to table and columns
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders IS
    'Cleaned orders table with data quality validations applied'
""")

spark.sql(f"""
    COMMENT ON COLUMN {CATALOG}.{BRONZE_SCHEMA}.orders.customer_id IS
    'Customer identifier - PII data, access restricted'
""")

### Add tags to orders table

You can classify and manage tables in Unity Catalog using **tags** (key-value pairs). Tags help with data discovery, compliance, and governance (e.g., marking tables as PII, GDPR, or Sensitive).

Example: 
sql
ALTER TABLE retailhub_trainer.bronze.orders
 SET TAGS ('pii' = 'false', 'data_classification' = 'transactional', 'retention' = '7_years');

- `pii`: Indicates if table contains personally identifiable information.
- `data_classification`: Describes the type of data (e.g., transactional, reference).
- `retention`: Specifies data retention policy.

You need `APPLY TAG` privilege to add tags.

In [0]:
# Add tags to orders table
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.orders 
    SET TAGS ('sensitivity' = 'high', 'domain' = 'sales')
""")

# Add tags to customer_id column
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.orders 
    ALTER COLUMN customer_id SET TAGS ('pii' = 'true')
""")

display(spark.createDataFrame([("Status", " Tags added to table and column")], ["Info", "Value"]))

## Querying Metadata and Tags

Exploring Unity Catalog metadata using system tables and information_schema to discover tags, comments, and permissions across your catalog.

In [0]:
# Find all columns marked as PII
pii_columns = spark.sql(f"""
    SELECT 
        catalog_name, 
        schema_name, 
        table_name, 
        column_name, 
        tag_value 
    FROM system.information_schema.column_tags
    WHERE tag_name = 'pii' AND tag_value = 'true'
      AND catalog_name = '{CATALOG}'
""")

display(pii_columns)

## Unity Catalog Functions (UDF)

Registering and managing reusable SQL and Python functions within Unity Catalog, with centralized access control and lineage tracking.

**Functions** in Unity Catalog allow:
- Creating reusable SQL/Python functions
- Centralized management of business logic
- Access control through GRANT/REVOKE
- Lineage tracking for functions

**Function types**:
- **Scalar Functions**: return a single value
- **Table Functions**: return a table
- **SQL Functions**: written in SQL
- **Python Functions**: written in Python (UDF)

### Demo: Data Classification (UDF + Tagging)

> *Note: This section covers data tagging for classification, related to governance.*

**Tagging** allows data classification (e.g., PII, Sensitive, GDPR) at the table or column level.
This facilitates data discovery and governance (e.g., reporting all tables containing personal data).

In [0]:
# SQL Function - masking customer_id
spark.sql(f"""
  CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_customer_id(customer_id STRING)
  RETURNS STRING
  LANGUAGE SQL
  COMMENT 'Masks customer_id, showing only last 3 digits'
  RETURN CONCAT('****', SUBSTRING(CAST(customer_id AS STRING), -3))
""")

In [0]:
# Test mask_customer_id function
result_df = spark.sql(f"""
  SELECT 
    customer_id,
    {CATALOG}.{SILVER_SCHEMA}.mask_customer_id(customer_id) as masked_id,
    first_name,
    last_name
  FROM {CATALOG}.{BRONZE_SCHEMA}.customers
  LIMIT 5
""")

display(result_df)

**Creating categorize_price function**

Python UDF function categorizes product prices:
- **Low**: < 50
- **Medium**: 50-200 
- **High**: > 200

Python UDF can contain any Python logic.

In [0]:
# Python UDF - price categorization
spark.sql(f"""
  CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.categorize_price(price DOUBLE)
  RETURNS STRING
  LANGUAGE PYTHON
  COMMENT 'Categorizes prices: Low, Medium, High'
  AS $$
    if price < 50:
        return "Low"
    elif price < 200:
        return "Medium"
    else:
        return "High"
  $$
""")

In [0]:
# Test categorize_price function
result_df = spark.sql(f"""
  SELECT 
    product_name,
    unit_cost,
    {CATALOG}.{SILVER_SCHEMA}.categorize_price(unit_cost) as price_category
  FROM {CATALOG}.{BRONZE_SCHEMA}.products
  ORDER BY unit_cost
  LIMIT 10
""")

display(result_df)

**Setting permissions for data-analysts**

The `data-analysts` group received:
- **USE CATALOG**: Access to catalog
- **USE SCHEMA**: Access to Silver schema 
- **SELECT**: Read data from Silver schema

**Setup:** Create groups for demonstration purposes
> **Note:** This requires account admin privileges. If you don't have them, ensure these groups exist.

* TO DO IN GUI

In [0]:
# Grant catalog access to data analysts
spark.sql(f"""
    GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`
""")

spark.sql(f"""
    GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`
""")

spark.sql(f"""
    GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`
""")

**Permissions for Data Analysts (Gold Layer):**

In [0]:
# GRANT for data-analysts on Gold schema
spark.sql(f"""
  GRANT USE SCHEMA ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`
""")

spark.sql(f"""
  GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`
""")

**Table-specific access control**

Fine-grained permissions:
- **finance-team**: Access to fact_sales (revenue analysis)
- **marketing-team**: Access to customers_masked (customer insights with PII masking)

In [0]:
# GRANT EXECUTE na Functions
spark.sql(f"""
  GRANT EXECUTE ON FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_customer_id TO `analysts`
""")

spark.sql(f"""
  GRANT EXECUTE ON FUNCTION {CATALOG}.{SILVER_SCHEMA}.categorize_price TO `analysts`
""")

In [0]:
# Verify permissions on table
spark.sql(f"""
    SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
""").display()

## Data Masking and Row-Level Security

Implementing column-level masking with dynamic views and row-level security to control data visibility based on user identity and group membership.

### Column-level masking (Dynamic Views):

Use `current_user()` and `is_account_group_member()` functions for conditional masking:

In [0]:
# Create masked view for PII data
spark.sql(f"""
  CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.customers_masked AS
  SELECT 
    customer_id,
    CASE 
      WHEN is_account_group_member('alt_test') THEN first_name
      ELSE CONCAT(LEFT(first_name, 1), '***')
    END as first_name,
    CASE 
      WHEN is_account_group_member('alt_test') THEN last_name
      ELSE CONCAT(LEFT(last_name, 1), '***')
    END as last_name,
    city,
    country,
    registration_date
  FROM {CATALOG}.{BRONZE_SCHEMA}.customers
""")

In [0]:
df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customers_masked")
display(df)

**View customers_masked created**

View with dynamic PII data masking:

In [0]:
# Test View z maskowaniem
result_df = spark.sql(f"""
  SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.customers_masked LIMIT 10
""")

display(result_df)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.customers_masked;")

In [0]:
%sql
CREATE FUNCTION customer_mask_test (customer_id STRING)
  RETURN CASE WHEN is_account_group_member('alt_test') THEN customer_id ELSE 'CUST-**-****' END

In [0]:
# Define the customers_masked table schema without CTAS
spark.sql(f"""
  CREATE OR REPLACE TABLE {CATALOG}.{SILVER_SCHEMA}.customers_masked (
    customer_id STRING,
    customer_id_masked STRING MASK customer_mask_test,
    first_name STRING,
    last_name STRING,
    email STRING,
    country STRING
  )
""")

In [0]:
spark.sql(f"""
  INSERT INTO {CATALOG}.{SILVER_SCHEMA}.customers_masked (customer_id, customer_id_masked, first_name, last_name, email, country)
  VALUES
    ('CUST001', 'CUST001', 'Alice', 'Smith', 'alice.smith@example.com', 'USA'),
    ('CUST002', 'CUST002', 'Bob', 'Johnson', 'bob.johnson@example.com', 'Canada'),
    ('CUST003', 'CUST003', 'Carol', 'Williams', 'carol.williams@example.com', 'UK')
""")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.customers_masked"))

In [0]:
spark.sql(
    f"""
    CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.orders_hashed AS
    SELECT 
        order_id,
        SHA2(CAST(customer_id AS STRING), 256) as customer_id_hash,
        product_id,
        quantity,
        total_amount,
        order_datetime
    FROM {CATALOG}.{BRONZE_SCHEMA}.orders
    """
)

display(
    spark.createDataFrame(
        [
            ("View", f"{CATALOG}.{GOLD_SCHEMA}.orders_hashed"),
            ("Masking", "customer_id → SHA2-256 hash"),
            ("Purpose", "Analysts can aggregate without revealing customer_id")
        ],
        ["Parameter", "Value"]
    )
)

**View orders_hashed created**

Customer_id is hashed using SHA2-256. This enables:
- **Analysts**: Data aggregation without revealing customer_id
- **Privacy**: Maintaining anonymity while preserving grouping capability
- **Compliance**: Meeting GDPR/privacy regulations requirements

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.orders_hashed"))

### Row-Level Security (RLS)

Restrict which rows users can see based on their identity or group membership:

**RLS View customers_rls created**

Row-Level Security filters data based on group membership:
- **global-access**: Sees all customers
- **east-coast-team**: Only customers from NY, NJ, NC, GA 
- **alt_test**: Only customers from CA
- **midwest-team**: Only customers from FL, IL, TX, MI
- **Other groups**: No access (FALSE)

Automatic row filtering without data duplication.

In [0]:
# Creating RLS view - access per region (state)
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.customers_rls AS
    SELECT *
    FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    WHERE 
        CASE 
            WHEN is_account_group_member('global-access') THEN TRUE
            WHEN is_account_group_member('east-coast-team') THEN UPPER(state) IN ('NY', 'NJ', 'NC', 'GA')
            WHEN is_account_group_member('alt_test') THEN UPPER(state) = 'CA'
            WHEN is_account_group_member('midwest-team') THEN UPPER(state) IN ('FL', 'IL', 'TX', 'MI')
            ELSE FALSE
        END
""")
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customers_rls"))
display(spark.createDataFrame([
    ("RLS View", f"{CATALOG}.{GOLD_SCHEMA}.customers_rls"),
    ("Mechanism", "Filtering per state based on group membership"),
    ("global-access", "All customers"),
    ("east-coast-team", "NY, NJ, NC, GA"),
    ("alt_test", "CA"),
    ("midwest-team", "FL, IL, TX, MI")
], ["Group", "Visibility"]))

**Granting permissions to RLS Views:**

### ABAC (Attribute-Based Access Control)

ABAC is a **security pattern** in Unity Catalog that combines multiple governance features to control access based on **data attributes** rather than just user identity.

**ABAC in Databricks = Tags + Column Masks + Row Filters**

| Component | Purpose | Mechanism |
|---|---|---|
| **Tags** (Data Classification) | Mark sensitive data with attributes | `ALTER TABLE ... SET TAGS ('pii' = 'true')` |
| **Column Masks** | Hide/transform column values based on user group | `CREATE FUNCTION mask_fn(...)` + Dynamic Views |
| **Row Filters** | Restrict which rows a user can see | RLS Views with `IS_ACCOUNT_GROUP_MEMBER()` |

**How ABAC works in practice:**

1. **Classify** -- Tag tables and columns with sensitivity levels (e.g., `pii`, `data_classification`)
2. **Define policies** -- Create masking functions and RLS views that enforce access rules
3. **Assign** -- Grant access to groups; the tags + functions automatically enforce attribute-based filtering
4. **Audit** -- Use `system.information_schema` to track tags and access patterns

**Key difference from RBAC:**
- **RBAC** (Role-Based): Access determined by user's *role* (e.g., `data-analysts` group gets SELECT)
- **ABAC** (Attribute-Based): Access determined by *data attributes* (e.g., columns tagged `pii=true` are automatically masked)

> **Pro Tip**: Unity Catalog implements ABAC through the combination of Tags, Column Masks, and Row Filters. Know that tags provide metadata for governance, while masks and filters enforce data-level security policies.

### Native Column Masks & Row Filters (ALTER TABLE)

Beyond view-based masking, Unity Catalog supports **native column masks** and **row filters** applied directly to tables. This is the preferred approach — security is enforced at the table level, regardless of how users query the data.

| Feature | View-Based (shown above) | Native (ALTER TABLE) |
|---------|------------------------|---------------------|
| Scope | Only when querying the view | Every query on the table |
| Setup | CREATE VIEW with CASE WHEN | CREATE FUNCTION + ALTER TABLE |
| Maintenance | Must update view if logic changes | Update function only |

**Column Masks & Row Filters — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Create mask function | `CREATE FUNCTION mask_email(email STRING) RETURN CASE WHEN is_account_group_member('admins') THEN email ELSE '***' END` |
| Apply column mask | `ALTER TABLE t ALTER COLUMN email SET MASK catalog.schema.mask_email` |
| Drop column mask | `ALTER TABLE t ALTER COLUMN email DROP MASK` |
| Create row filter | `CREATE FUNCTION filter_fn(status STRING) RETURN is_account_group_member('admins') OR status = 'active'` |
| Apply row filter | `ALTER TABLE t SET ROW FILTER catalog.schema.filter_fn ON (status)` |
| Drop row filter | `ALTER TABLE t DROP ROW FILTER` |

#### Step 1: Create a Masking Function

A SQL UDF that returns the masked value. It receives the original column value and can check group membership.

In [ ]:
# Create a masking function for email column
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{GOLD_SCHEMA}.mask_email_native(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('admins') THEN email
        ELSE CONCAT('***@', SUBSTRING_INDEX(email, '@', -1))
    END
""")

print("Masking function created: mask_email_native")

#### Step 2: Apply Mask to a Column with ALTER TABLE

`SET MASK` attaches the function directly to the column. Every query on this table will automatically apply the mask.

In [ ]:
# Apply mask to the email column
spark.sql(f"""
    ALTER TABLE {CATALOG}.{GOLD_SCHEMA}.customers
    ALTER COLUMN email SET MASK {CATALOG}.{GOLD_SCHEMA}.mask_email_native
""")

# Verify — non-admins will see masked emails
display(spark.sql(f"SELECT customer_id, first_name, email FROM {CATALOG}.{GOLD_SCHEMA}.customers LIMIT 5"))

#### Step 3: Create and Apply a Row Filter

Row filters restrict which rows are visible to different users. Like column masks, they are applied directly to the table.

In [ ]:
# Create a row filter function — admins see all, others see only 'active' customers
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{GOLD_SCHEMA}.filter_active_customers(status STRING)
    RETURNS BOOLEAN
    RETURN IF(IS_ACCOUNT_GROUP_MEMBER('admins'), true, status = 'active')
""")

# Apply row filter to the table
spark.sql(f"""
    ALTER TABLE {CATALOG}.{GOLD_SCHEMA}.customers
    SET ROW FILTER {CATALOG}.{GOLD_SCHEMA}.filter_active_customers ON (status)
""")

# Verify — non-admins will only see 'active' customers
display(spark.sql(f"SELECT customer_id, first_name, status FROM {CATALOG}.{GOLD_SCHEMA}.customers LIMIT 10"))

#### Step 4: Cleanup — Removing Masks and Filters

Use `DROP MASK` and `DROP ROW FILTER` to remove security policies from a table.

In [ ]:
# Remove column mask
spark.sql(f"""
    ALTER TABLE {CATALOG}.{GOLD_SCHEMA}.customers
    ALTER COLUMN email DROP MASK
""")

# Remove row filter
spark.sql(f"""
    ALTER TABLE {CATALOG}.{GOLD_SCHEMA}.customers
    DROP ROW FILTER
""")

print("Column mask and row filter removed successfully")

## Access Management: GRANT / REVOKE

Managing permissions in Unity Catalog using GRANT and REVOKE statements, with privilege inheritance across the catalog-schema-table hierarchy.

### Privileges Hierarchy in Unity Catalog:

**Privilege levels**:
1. **Metastore-level**: CREATE CATALOG, USE CATALOG
2. **Catalog-level**: USE CATALOG, CREATE SCHEMA
3. **Schema-level**: USE SCHEMA, CREATE TABLE, CREATE FUNCTION, CREATE VOLUME
4. **Object-level**: SELECT, MODIFY (INSERT/UPDATE/DELETE/MERGE), EXECUTE

**Securable Objects - Inheritance**:
- Privileges inherit down the hierarchy
- GRANT on Catalog → inherits to all Schemas and Tables
- GRANT on Schema → inherits to all Tables in that Schema
- You can grant privileges at specific level for fine-grained control

### GRANT/REVOKE Examples:

**GRANT / REVOKE — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Grant on table | `GRANT SELECT ON TABLE catalog.schema.table TO \`group\`` |
| Grant on schema | `GRANT USE SCHEMA ON SCHEMA catalog.schema TO \`group\`` |
| Grant on catalog | `GRANT USE CATALOG ON CATALOG catalog_name TO \`group\`` |
| Revoke | `REVOKE SELECT ON TABLE catalog.schema.table FROM \`group\`` |
| Show grants | `SHOW GRANTS ON TABLE catalog.schema.table` |
| Key privileges | `SELECT` / `MODIFY` / `CREATE` / `EXECUTE` / `ALL PRIVILEGES` |

In [0]:
# GRANT access to customers_rls
spark.sql(
    f"""
    GRANT SELECT ON VIEW {CATALOG}.{GOLD_SCHEMA}.customers_rls TO `account users`
    """
)

**Granting permissions to orders_hashed**

In [0]:
# GRANT access to orders_hashed
spark.sql(f"""
  GRANT SELECT ON VIEW {CATALOG}.{GOLD_SCHEMA}.orders_hashed TO `account users`
""")

**Revoking access to base tables (Enforcement):**

In [0]:
# Revoke direct access to base table
spark.sql(f"""
    REVOKE SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders FROM `account users`
""")

**RLS Views - Access control setup**

Security pattern:
1. **GRANT SELECT** on RLS Views for `all-users`
2. **REVOKE SELECT** on base tables (force Views usage)
3. **Automatic filtering** based on group membership

Users can SELECT from Views, but not from base tables - enforcing RLS.

## Data Lineage and Audit Logging

Tracking data flow and access patterns across the lakehouse using Unity Catalog's automatic lineage and system audit logs.

### Querying Data Lineage:

Unity Catalog automatically tracks lineage for:
- Table → Table (ETL transformations)
- Notebook → Table (data writes)
- Dashboard → Table (BI queries)
- ML Model → Table (training data)

**General Table Lineage**

In [0]:
# Query table lineage from system tables
lineage_df = spark.sql(f"""
  SELECT 
    source_table_full_name,
    source_type,
    target_table_full_name,
    target_type,
    event_date,
    created_by
  FROM system.access.table_lineage
  WHERE target_table_full_name LIKE '{CATALOG}.%'
  ORDER BY event_date DESC
  LIMIT 50
""")

display(lineage_df)

**Lineage for tables in catalog displayed**

The system automatically tracks lineage for:
- **Table → Table**: ETL transformations
- **Notebook → Table**: Data writes 
- **Dashboard → Table**: BI queries
- **ML Model → Table**: Training data

Lineage is available through `system.access.table_lineage` without additional instrumentation.

**1. Upstream Lineage (Sources)**

In [0]:
# Find upstream dependencies (sources) for a table
upstream_df = spark.sql(f"""
    SELECT DISTINCT
        source_table_full_name,
        source_type
    FROM system.access.table_lineage
    WHERE target_table_full_name = '{CATALOG}.{GOLD_SCHEMA}.fact_sales'
""")

display(upstream_df)

**^ Upstream: Source tables for fact_sales**

Shows all tables used as data sources in the `fact_sales` View. Helpful for impact analysis when making changes to upstream tables.

**2. Downstream Lineage (Consumers)**

In [0]:
# Find downstream dependencies (consumers) of a table
downstream_df = spark.sql(f"""
    SELECT DISTINCT
        target_table_full_name,
        target_type
    FROM system.access.table_lineage
    WHERE source_table_full_name = '{CATALOG}.{BRONZE_SCHEMA}.customers'
""")

display(downstream_df)

**Downstream: Views/Tables consuming customers**

Shows all Views and tables that consume data from the `customers` table. Critical for understanding impact of changes and data governance.

**3. Column-Level Lineage**

In [0]:
# Column-level lineage (if available)
column_lineage = spark.sql(f"""
    SELECT 
        source_table_full_name,
        source_column_name,
        target_table_full_name,
        target_column_name,
        event_date
    FROM system.access.column_lineage
    WHERE target_table_full_name = '{CATALOG}.{GOLD_SCHEMA}.fact_sales'
    ORDER BY target_column_name
""")

display(column_lineage)

**Column-level lineage for fact_sales**

Unity Catalog tracks lineage at column level - which columns in source tables affect which columns in the target table. Detailed information for data governance and impact analysis.

### Audit Logging

Unity Catalog logs all access and operations:

**1. General Audit Logs**

In [0]:
# Query audit logs
audit_df = spark.sql("""
    SELECT 
        event_time,
        user_identity.email as user_email,
        service_name,
        action_name,
        request_params.full_name_arg as table_name,
        response.status_code,
        request_id
    FROM system.access.audit
    WHERE action_name IN ('getTable', 'createTable', 'deleteTable', 'updateTable')
        AND event_date >= current_date() - INTERVAL 7 DAYS
    ORDER BY event_time DESC
    LIMIT 100
""")
audit_df.display()

**2. Sensitive Data Access**

In [0]:
# Track who accessed sensitive tables
sensitive_access = spark.sql(f"""
    SELECT 
        event_time,
        user_identity.email as user,
        action_name,
        request_params.full_name_arg as table_accessed,
        source_ip_address
    FROM system.access.audit
    WHERE request_params.full_name_arg LIKE '{CATALOG}.%.customers%'
        AND action_name = 'getTable'
        AND event_date >= current_date() - INTERVAL 7 DAYS
    ORDER BY event_time DESC
    LIMIT 100
""")

display(sensitive_access)

** Audit logs: Access to customers table (last 7 days)**

Monitoring access to sensitive tables with PII data:
- **Who**: User email
- **When**: Event time 
- **What**: Table name
- **From where**: Source IP address

Critical for compliance (GDPR, HIPAA) and security monitoring.

**3. Privilege Changes**

In [0]:
# Grant/Revoke audit trail
grant_audit = spark.sql("""
    SELECT 
        event_time,
        user_identity.email as admin_user,
        action_name,
        request_params.privilege as privilege_granted,
        request_params.securable_full_name as object_name,
        request_params.principal as grantee
    FROM system.access.audit
    WHERE action_name IN ('grantPrivilege', 'revokePrivilege')
        AND event_date >= current_date() - INTERVAL 30 DAYS
    ORDER BY event_time DESC
""")

display(grant_audit)

**Audit trail of privilege changes**

Complete audit trail of permission changes:
- **Admin user**: Who executed GRANT/REVOKE
- **Action**: grantPrivilege or revokePrivilege
- **Privilege**: Which permission (SELECT, MODIFY, etc.)
- **Object**: On which object (table, schema, catalog)
- **Grantee**: To whom permissions were granted/revoked

Essential for governance and compliance audits.

## Delta Sharing

Securely sharing data across organizations and cloud platforms using the open Delta Sharing protocol, with fine-grained access control.

### Components:
- **Share**: collection of tables to share
- **Recipient**: organization/user receiving data
- **Provider**: data owner (you)

### Create Share:

**2025-2026 Enhancements:**
- **Iceberg compatibility** — Recipients can read shared data as Apache Iceberg tables
- **Schema sharing** — Share entire schemas (not just individual tables)
- **Recipient access tokens** — Improved security with short-lived tokens
- **Audit integration** — All sharing activity logged in `system.access.audit`

**2025-2026 Enhancements:**
- **Iceberg compatibility** — Recipients can read shared data as Apache Iceberg tables
- **Schema sharing** — Share entire schemas (not just individual tables)
- **Recipient access tokens** — Improved security with short-lived tokens
- **Audit integration** — All sharing activity logged in `system.access.audit`

**2025-2026 Enhancements:**
- **Iceberg compatibility** — Recipients can read shared data as Apache Iceberg tables
- **Schema sharing** — Share entire schemas (not just individual tables)
- **Recipient access tokens** — Improved security with short-lived tokens
- **Audit integration** — All sharing activity logged in `system.access.audit`

In [0]:
# Creating Share for external partners
share_name = f"{CATALOG}_partner_share"

spark.sql(f"""
  CREATE SHARE IF NOT EXISTS {share_name}
  COMMENT 'Data sharing for business partners'
""")

**Share '{share_name}' created**

Delta Sharing Share is a collection of tables for secure sharing with external partners:
- **Cross-org**: Between different Databricks organizations
- **Cross-cloud**: AWS ↔ Azure ↔ GCP 
- **Open protocol**: Open-source standard

In [0]:
# Add table to Share (Gold layer only - aggregated data)
spark.sql(f"""
  ALTER SHARE {share_name}
  ADD TABLE {CATALOG}.{GOLD_SCHEMA}.fact_sales
""")

In [0]:
spark.sql(f"""
  ALTER SHARE {share_name}
  ADD SCHEMA {CATALOG}.{SILVER_SCHEMA}
""")

**Table fact_sales added to Share**

Best practice: Share only Gold layer (aggregated data):
- **Security**: No access to raw data
- **Privacy**: Aggregations hide individual records
- **Stability**: Gold layer has stable schema and structure

**Tables in Share verified**

Share currently contains the added tables and can be shared with recipients. Recipients will receive an activation link to consume shared data via Delta Sharing protocol.

In [0]:
# Verify Share contents
spark.sql(f"SHOW ALL IN SHARE {share_name}").display()

### Create Recipient

> **[UI DEMO]** Create a recipient in the Databricks UI: Catalog -> Delta Sharing -> New Recipient.

### Consuming shared data (as recipient)

> **[UI DEMO]** As a recipient, use the activation link to access shared data via Open Sharing protocol.

### Best practices for Delta Sharing

1. **Share only aggregated/gold data**: don't share raw/bronze layers
2. **Use views for masking**: create view with masked PII before sharing
3. **Monitor access**: track who accesses shared data
4. **Version control**: use table versions for stable APIs
5. **Documentation**: clear documentation for recipients

> **SKIP IF SHORT ON TIME** — INFORMATION_SCHEMA queries can be explored in the workshop.

## Information Schema

Unity Catalog provides an `INFORMATION_SCHEMA` in every catalog for querying metadata about tables, columns, permissions, and schemas using standard SQL.

```sql
-- List all tables in a schema
SELECT table_name, table_type, created
FROM my_catalog.information_schema.tables
WHERE table_schema = 'my_schema';

-- List all columns for a table
SELECT column_name, data_type, is_nullable
FROM my_catalog.information_schema.columns
WHERE table_name = 'customers';

-- Check grants on a table
SELECT grantee, privilege_type
FROM my_catalog.information_schema.table_privileges
WHERE table_name = 'customers';
```

**Available views in `information_schema`:**

| View | Content |
|------|--------|
| `tables` | All tables and views |
| `columns` | Column definitions |
| `table_privileges` | Granted permissions |
| `schemata` | Schema metadata |
| `catalogs` | Catalog information |
| `views` | View definitions |

**Pro Tip:** `INFORMATION_SCHEMA` is the standard SQL way to query metadata. It is available per catalog in Unity Catalog.

## Troubleshooting

Common Unity Catalog permission and access issues with their solutions, useful for debugging governance configurations.

### Problem 1: "Table or view not found"
**Cause**: Missing USE CATALOG or USE SCHEMA permissions 
**Solution**:
```sql
GRANT USE CATALOG ON CATALOG <catalog_name> TO <principal>;
GRANT USE SCHEMA ON SCHEMA <catalog>.<schema> TO <principal>;
```

### Problem 2: "Permission denied" on SELECT
**Cause**: Missing SELECT permissions on table 
**Solution**:
```sql
GRANT SELECT ON TABLE <catalog>.<schema>.<table> TO <principal>;
-- or on entire schema:
GRANT SELECT ON SCHEMA <catalog>.<schema> TO <principal>;
```

### Problem 3: "Cannot execute function"
**Cause**: Missing EXECUTE permission on function 
**Solution**:
```sql
GRANT EXECUTE ON FUNCTION <catalog>.<schema>.<function_name> TO <principal>;
```

### Problem 4: "Volume not accessible"
**Cause**: Missing READ VOLUME / WRITE VOLUME permissions 
**Solution**:
```sql
GRANT READ VOLUME ON VOLUME <catalog>.<schema>.<volume> TO <principal>;
GRANT WRITE VOLUME ON VOLUME <catalog>.<schema>.<volume> TO <principal>;
```

### Problem 5: RLS View not filtering data
**Cause**: User doesn't belong to any group defined in CASE WHEN 
**Solution**: Add user to appropriate group or add default fallback in View

### Problem 6: Lineage not showing dependencies
**Cause**: Lineage is automatic but may be delayed by a few minutes 
**Solution**: Wait 5-10 minutes and query system.access.table_lineage again

### Problem 7: Share not visible to recipient
**Cause**: Recipient hasn't activated the activation link 
**Solution**: Send activation link from DESCRIBE RECIPIENT

> **SKIP IF SHORT ON TIME** — Lakehouse Federation is an advanced topic for later study.

## Lakehouse Federation

**Lakehouse Federation** allows Unity Catalog to query external databases (PostgreSQL, MySQL, Snowflake, BigQuery, SQL Server, etc.) without copying data — using **Foreign Catalogs**.

```
Databricks Workspace
  └── Unity Catalog
        └── FOREIGN CATALOG  ← maps to external database
              └── FOREIGN SCHEMA
                    └── FOREIGN TABLE  ← read-only view of external data
```

### How it works

1. **Create a Connection** → credentials for the external system
2. **Create a Foreign Catalog** → maps the external database into Unity Catalog namespace
3. **Query via SQL** → `SELECT * FROM foreign_catalog.schema.table` — Spark pushes down predicates to the source

| Feature | Details |
|---|---|
| Supported sources | PostgreSQL, MySQL, Snowflake, BigQuery, SQL Server, Hive, Redshift |
| Access control | Standard Unity Catalog GRANT/REVOKE applies |
| Data movement | None — queries run at the source (federated execution) |
| Write support | Read-only by default |

> **Pro Tip:** Lakehouse Federation uses `CREATE CONNECTION` + `CREATE FOREIGN CATALOG`. It is **different from Delta Sharing** — federation is about querying external systems, Delta Sharing is about sharing Delta tables with external consumers.

In [ ]:
# NOTE: The commands below require a real external database connection.
# They are shown as examples — uncomment and adapt for your environment.

# ── Step 1: Create a Connection (credentials to the external DB) ─────────────
# spark.sql("""
#   CREATE CONNECTION IF NOT EXISTS my_postgres_conn
#   TYPE POSTGRESQL
#   OPTIONS (
#     host  'postgres.example.com',
#     port  '5432',
#     user  secret('my_scope', 'pg_user'),
#     password secret('my_scope', 'pg_password')
#   )
# """)

# ── Step 2: Create a Foreign Catalog ─────────────────────────────────────────
# spark.sql("""
#   CREATE FOREIGN CATALOG IF NOT EXISTS pg_sales
#   USING CONNECTION my_postgres_conn
#   OPTIONS (database 'salesdb')
# """)

# ── Step 3: Query the external data through Unity Catalog ─────────────────────
# display(spark.sql("SELECT * FROM pg_sales.public.orders LIMIT 10"))

# ── Grant access to the foreign catalog (same as any catalog) ─────────────────
# spark.sql("GRANT USE CATALOG ON FOREIGN CATALOG pg_sales TO `data-analysts`")
# spark.sql("GRANT SELECT ON ALL TABLES IN SCHEMA pg_sales.public TO `data-analysts`")

# Federation summary table
display(spark.createDataFrame([
    ("CREATE CONNECTION",      "Stores external DB credentials (uses Secrets)"),
    ("CREATE FOREIGN CATALOG", "Maps external DB into UC namespace"),
    ("GRANT on FOREIGN CATALOG","Same RBAC as regular catalogs"),
    ("SELECT on foreign table", "Federated query — data stays at source"),
    ("vs Delta Sharing",        "Sharing = export; Federation = in-place query"),
], ["Command / Concept", "Description"]))

## Best Practices

Key recommendations for organizing catalogs, managing access control, implementing data masking, and monitoring your Databricks governance environment.

### 1. **Catalog Organization**
- Use environment-based catalogs: `dev`, `test`, `prod`
- Organize schemas by layers: `bronze`, `silver`, `gold`
- Apply naming conventions: `<catalog>.<schema>.<object>`

### 2. **Access Control**
- **Principle of Least Privilege**: Grant minimum required permissions
- Use groups, not individual users
- Inheritance: GRANT on Catalog → inherits to Schema → inherits to Tables
- Regularly audit permissions (SHOW GRANTS)

### 3. **Data Masking & RLS**
- Mask PII in Views for users without pii-access-group
- Use RLS for multi-tenant scenarios
- Always test masking with different group memberships

### 4. **Lineage & Audit**
- Leverage automatic lineage to track data flow
- Regularly check audit logs for sensitive tables
- Monitor lineage after pipeline changes

### 5. **Delta Sharing**
- Share only Gold layer (aggregated data)
- Use masked Views in Share
- Document Share contracts for recipients

### 6. **Documentation & Governance**
- Add COMMENT to all tables, views, functions
- Use Table Properties for metadata (owner, PII, retention)
- Regularly check governance health checks

### 7. **Monitoring & Observability**
- Use `system.billing.usage` for cost tracking and chargeback
- Monitor job/pipeline SLAs with `system.lakeflow.job_run_timeline`
- Track slow queries in `system.query.history`
- Set up daily governance health checks (undocumented tables, untagged PII)
- Create alerts for cost spikes and job failures

## Summary

### Cost Management

| Topic | Key Concept |
|---|---|
| DBU | Billing unit — varies by compute type and SKU |
| OPTIMIZE | Fewer files → lower compute cost per query |
| System Tables | `system.billing.usage`, `system.compute.clusters` |
| Job Cluster | Always cheaper than All-purpose for production |
| Serverless | Instant start, per-query billing, zero idle cost |

### Security & Governance

| Topic | Key Concept |
|---|---|
| Unity Catalog | 3-level namespace: catalog.schema.table |
| Column Masking | `ALTER TABLE … ALTER COLUMN … SET MASK` |
| Row Filter | `ALTER TABLE … SET ROW FILTER` |
| Audit Logs | `system.access.audit` — all access events |
| Data Lineage | `system.access.table_lineage` — upstream / downstream |
| Delta Sharing | Open protocol for cross-org/cloud data sharing |

← [03 — Optimization & Maintenance](03_optimization_demo.ipynb) | **[README](../../../README.md)**